## Statistische Dataverwerking - Labosessie 7: Data Manipulation with Pandas  ##


In this lab session, we work with Jupyter Notebook and pandas for data manipulation. Before you start, read through the following notes:
- You can execute an activated Python cell by clicking the Play button or use the hot keys *CTRL + ENTER*
- The autocomplete popup should appear automatically. You can close the popup with *ESC* and re-opening it with *TAB*
- If the autocomplete popup does not appear, you might need to activate it: Settings > Settings Editor > Code Completion > Enable autocompletion.
- Always check with the [official documentation](https://pandas.pydata.org/docs/) if you do not understand a function or its parameters.

Check first the version of pandas you are working with by executing the code cell below.

In [ ]:
!pip show pandas

## 0. Preparing the notebook ##
We import **pandas** with its standard alias *pd*. We access pandas elements via pd.\[element name\].


In [ ]:
import pandas as pd

# Display all DataFrame columns
pd.set_option('display.max_colwidth', None)

## 1. Creating a DataFrame
We consider three scenarios to create a DataFrame:
1. Create a DataFrame from a Python dictionary
2. Create a DataFrame from a file
3. Create a DataFrame from a SQL result set

In [ ]:
### 1.1 Create a DataFrame from a Python dictionary
df_dict = pd.DataFrame({
    'letter': [ 'a', 'b', 'c', 'd', 'e' ], 
    'number': [1, 2, 3, 4, 5],
    'anum': ['a', 2.0, 'c', 4.0, 'e']
})

df_dict

In [ ]:
### 1.2 Create a DataFrame from a file
df_file = pd.read_csv('data/three_hour_movies.tsv', 
                        sep='\t', 
                        encoding='utf8'
                       )
### Export a DataFrame as a file
# df_file.to_csv('data/three_hour_movies(copy).tsv', sep='\t', encoding='utf8', index=False)
df_file

In [ ]:
### 1.3 Create a DataFrame from a SQL result set
from sqlalchemy import create_engine

### Settings 
user = 'q1234567' # Update your q-number
password = user
host = '10.64.50.77'
port = '5432'
database_name = 'postgres'
driver = 'postgresql+psycopg2'

connection_string = f'{driver}://{user}:{password}@{host}:{port}/{database_name}'
print('My connection string:', connection_string)
engine = create_engine(connection_string)
### Settings End

sql_string = "select tb.* from imdb.title_basics tb limit 5"
df_sql = pd.read_sql_query(sql_string, con=engine)
df_sql

## 2. Analyzing a DataFrame

1. Return all column names
2. Return all column name with datatypes
3. Return row and column count
4. Return the descriptive statistics for numerical columns
5. Return the descriptive statistics for categorical columns

In [ ]:
### 2.1 Return all column names
df_sql.columns

In [ ]:
### 2.2 Return all column name with datatypes
df_sql.dtypes

In [ ]:
### 2.3 Return row and column count
df_sql.shape

In [ ]:
### 2.4 Return the descriptive statistics for numerical columns
df_sql.describe()

In [ ]:
### 2.5 Return the descriptive statistics for categorical columns
df_sql.describe(include=['object', 'category', 'str'])

## 3. Analyze a Series ("Column")
1. Check the datatype of the series
2. Check the unique values of a series
3. Detect duplicated values
4. Count the null values in a series
5. Get basic descriptive statistics
6. Count occurrences including NaNs
7. Get minimal value of a numerical series

In [ ]:
### 3.1 Check the datatype of the series
s = df_sql['startyear']
s.dtype

In [ ]:
### 3.2 Check the unique values of a series
s.unique()

In [ ]:
### 3.3 Detect duplicated values
s.duplicated().sum()

In [ ]:
### 3.4 Count the null values in a series
s.isna().sum()

In [ ]:
### 3.5 Get basic descriptive statistics
pd.DataFrame(s.describe())

In [ ]:
### 3.6 Count occurrences including NaNs
s.value_counts(dropna=False)

In [ ]:
### 3.7 Get minimal value of a numerical series
s.min()

## 4. CRUD Operations on DataFrames
### Create
1. Add a column (Series) to a DataFrame
2. Add multiple columns
3. Add a row

### Read
4. Filter a DataFrame
5. Select columns

### Update
6. Multiply a series (update column)
7. Update values by condition
8. Convert a numeric series to string and back to numeric

### Delete
9. Drop a column (Series)
10. Remove a row
11. Drop rows by condition

In [ ]:
### 4.1 Add a column (Series) to a DataFrame
df = df_sql[:]
df["new column"] = s
df

In [ ]:
### 4.2 Add multiple columns
df[["copy title", "copy runtime"]] = df[["primarytitle", "runtimeminutes"]]
df

In [ ]:
### 4.3 Add a row
new_row = 	["tt123456",	"movie",	"De Nayer Chronicles",	"De Nayer Chronicles",	2026,	2026 , 120,	"Comedy, Drama",	2026, "De Nayer Chronicles"	,	120]
df.loc[len(df)] = new_row
df

In [ ]:
### 4.4 Filter a DataFrame
df[df["runtimeminutes"] > 100]

In [ ]:
### 4.5 Select columns
df[["primarytitle", "runtimeminutes"]]

In [ ]:
### 4.6 Multiply a series (update column)
df["watch twice"] = df["runtimeminutes"] * 2
df

In [ ]:
### 4.7 Update values by condition
df.loc[df["genres"].str.startswith("Drama"), "titletype"] = "DramaMovie"
df

In [ ]:
### 4.8  Convert a numeric series to string and back to numeric
df["str_runtimeminutes"] = df["runtimeminutes"].astype(str)
df["int_runtimeminutes"] = pd.to_numeric(df["str_runtimeminutes"], errors="coerce")
df.dtypes

In [ ]:
### 4.9 Drop a column (Series)
df.drop(columns=["new column", "copy title", "copy runtime", "watch twice", "str_runtimeminutes",	"int_runtimeminutes"])

In [ ]:
### 4.10 Remove a row
df.drop(index=2)

In [ ]:
### 4.11 Drop rows by condition
df[df["titletype"] != "movie"]

In [ ]:
### 4.12 Normalize a column
col = "runtimeminutes"
df[col + "_norm"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
df

## 5. A few practice examples
Before we begin, we import relevant classes and functions from packages:
- The **IPython** package contains functions to improve the usability with Jupyter notebooks. We import *display*, and *HTML* to prettify the output of executed cells.
- We import **pandas** with its standard alias *pd*. We access pandas elements via pd.\[element name\].
- From the **checker** package, we import the **Answer** class. Answer instances save DataFrames and Figures as MD5 hashes. We can compare hashes to check for identical results.

In [ ]:
from IPython.display import display, HTML
import pandas as pd
from sqlalchemy import create_engine
from checker import Answer

### Settings 
user = 'q1234567' # Update your q-number
password = user
host = '10.64.50.77'
port = '5432'
database_name = 'postgres'
driver = 'postgresql+psycopg2'

connection_string = f'{driver}://{user}:{password}@{host}:{port}/{database_name}'
print('My connection string:', connection_string)
engine = create_engine(connection_string)
### Settings End

# function to prettify HTML formatted output
show = lambda text: display(HTML(text))

# a = my_answers = Answer(ignore_check=True)
a = my_answers = Answer('hashes/manipulation-hashes.json')

### 5.1 IMDb Example
We retrieve the primarytitles, the runtime, the average rating, and the number of votes from the IMDb database.
The resulting DataFrame is directly saved as a member of the Answer instance.
The Display section at the end is for checking your result with the solution hash and also displays your result.
You never modify the Display section. Nonetheless, you should look at the Display section to see which member is checked and thus you probably must assign.
In this case, a.imdb is asked for in the Display section, so you must create the member beforehand.
The code sample below is already complete, and you can directly execute it.

In [ ]:
sql_string = """
    select tb.primarytitle, tb.runtimeminutes, tr.averagerating, tr.numvotes
    from imdb.title_basics tb
    join imdb.title_ratings tr on tb.tconst = tr.tconst
    order by tb.primarytitle, tb.runtimeminutes, tr.averagerating, tr.numvotes
"""

a.imdb = pd.read_sql_query(sql_string, con=engine)

### DISPLAY
show(a.check('imdb'))
display(a.imdb)

### 5.1.1 Add a column 'runtimehours' to a.imdb_hour which contains the runtime in hours.

In [ ]:
a.imdb_hour = a.imdb.copy()
a.imdb_hour['runtimehours'] = 

### DISPLAY
show(a.check('imdb_hour'))
display(a.imdb_hour)

### 5.1.2 Drop the column 'runtimeminutes' from a.imdb

In [ ]:
a.imdb_drop_min =

### DISPLAY
show(a.check('imdb_drop_min'))
display(a.imdb_drop_min)

### 5.1.3 Normalize the numvotes column in a new column 'numvotes_norm' in a.imdb_norm

In [ ]:
a.imdb_norm = a.imdb.copy()
col = "numvotes"
a.imdb_norm[col + "_norm"] = 

### DISPLAY
show(a.check('imdb_norm'))
display(a.imdb_norm)

### 5.1.4 Drop all rows with an index greater than 20 from a.imdb

In [ ]:
a.imdb_index = 

### DISPLAY
show(a.check('imdb_index'))
display(a.imdb_index)

### 5.1.5 Filter a.imdb that only rows remain where the title starts with 'T' and there are more than 300.000 votes.

In [ ]:
a.imdb_filtered = 

### DISPLAY
show(a.check('imdb_filtered'))
display(a.imdb_filtered)

### 5.2 Transfermarkt Example
Similar to 5.1., the first cell is already completed and you can execute it.

In [ ]:
sql_string = """
    select tp."Name", tp."Team", tp."Value", tp."Value last updated"   
    from transfermarkt.tm_player tp 
    where tp."Nationality" = 'Belgium'
    and tp."Value last updated" <> '' 
    order by tp."Value"
"""

a.tm = pd.read_sql_query(sql_string, con=engine)

### DISPLAY
show(a.check('tm'))
display(a.tm)

### 5.2.1 Create a new column 'last updated year' for a.tm_year which contains the year as integer from the column 'Value last updated'. 

In [ ]:
a.tm_year = a.tm.copy()



### DISPLAY
show(a.check('tm_year'))
display(a.tm_year)

### 5.2.2 There is a duplicate row in a.tm_year for 'Arnaud Dony'. Drop the duplicate.

In [ ]:
a.tm_drop_dupl = 

### DISPLAY
show(a.check('tm_drop_dupl'))
display(a.tm_drop_dupl)

### 5.3 FBref Example
Similar to 5.2., the first cell is already completed and you can execute it.

In [ ]:
sql_string = """
    select l."Rk", s."Squad" , p."Player"    
    from fbref.league l 
    join fbref.squad s on l."Season" = s."Season" and l."TeamID" = s."TeamID"
    join fbref.player p on p."Season" = s."Season"  and l."Top Scorer" = p."PlayerID" 
    where trim(l."Season") = '2025-2026'
    order by  l."Rk"  asc
"""

a.fb = pd.read_sql_query(sql_string, con=engine)

### DISPLAY
show(a.check('fb'))
display(a.fb)

### 5.3.1 Add a new column to a.fb_rank which contains the Rk value as integer. 

In [ ]:
a.fb_rank = a.fb.copy()

a.fb_rank['rank'] = 

### DISPLAY
show(a.check('fb_rank'))
display(a.fb_rank)

### 5.3.2 Drop the Rk column from a.fb_rank

In [ ]:
a.fb_drop_rk = 

### DISPLAY
show(a.check('fb_drop_rk'))
display(a.fb_drop_rk)

### 5.3.3 Rename the column 'Player' to 'Top Scorer' for a.fb_drop_rk

In [ ]:
a.fb_rename = 

### DISPLAY
show(a.check('fb_rename'))
display(a.fb_rename)

### 5.3.4 Create a DataFrame of the descriptive statistics of a.fb_rename including numerical and categorical columns.

In [ ]:
a.fb_describe = 

### DISPLAY
show(a.check('fb_describe'))
display(a.fb_describe)

In [ ]:
if a._ignore_check:
    a.hash_to_file('manipulation-hashes.json')

### Final Check
See if you have passed all checks.

In [ ]:
if not a._ignore_check:
    checks = a.get_checks()
    show(f'<h3>Passed Checks: {checks.Result.sum()}/{len(checks)}</h3>') 
    display(checks)